**By Rupesh Panwar ([VerBodh](https://verbodh.com))** · [LinkedIn](https://www.linkedin.com/in/rupesh-panwar/)

VerBodh is an AI and LLM apps startup, seeded in Singapore.

**How this build differs from the course's default route**

1. **Amazon Cognito for identity and tokens.** Handlers sign in to the dashboard through a Cognito user pool, and the
   app checks Cognito's tokens, in place of Supabase Auth.
2. **Amazon RDS for PostgreSQL in place of Supabase.** The claims data stays inside the customer's AWS account and
   never leaves the AWS ecosystem.
3. **No IAM users anywhere.** The customer doesn't allow long-lived IAM user keys, so every human call in this
   solution, this notebook and the agent's calls to the Gateway and the Memory included, signs in through
   AWS IAM Identity Center, with SSO and MFA. The AWS services themselves use IAM roles.

# Week 3 Day 3 challenge: the Episodic strategy

The challenge from the end of `labs/week3/week3day3.ipynb`: add AgentCore's Episodic strategy to the Memory, with its
own namespace for each customer, run Oliver Grant's two claims again, and compare what the Episodic strategy keeps
with the Semantic facts and the Summary.

**What this notebook does**

1. Makes a second Memory, `undercover_episodic`, with all three strategies. The lab's own Memory stays as it is.
2. Points the agent at it in `.env` for the two runs, and puts the lab's Memory back at the end.
3. Reseeds, then runs CLM-1005 and CLM-1006 with the agent on your laptop, against the app on Lambda, as Part 5 does.
4. Lists everything the Memory kept about Oliver, by strategy, and runs the agent's own search.
5. Deletes the `undercover_episodic` Memory.

**Before you run it:** finish Parts 1 to 4 of the Day 3 lab and don't run its Part 6 teardown yet. This notebook
uses the app on Lambda, the Gateway and the RDS or Supabase database the lab set up. It costs a few cents: two claim
runs on OpenRouter, and AgentCore's own model calls to extract the memories.

## Setup

In [ ]:
import truststore

# Trust the certificates in the Windows certificate store, so HTTPS works behind a network that inspects traffic.
truststore.inject_into_ssl()

import os
import shutil
import subprocess
import time
from pathlib import Path

import boto3
from dotenv import load_dotenv, set_key

In [ ]:
REPO = Path("../..").resolve()
ENV_FILE = REPO / ".env"
APP_FUNCTION = "undercover-app"
EPISODIC_MEMORY_NAME = "undercover_episodic"
PROJECT_TAG = {"project": "undercover"}
CUSTOMER = "CUS-3005"
CLAIMS = ["CLM-1005", "CLM-1006"]

load_dotenv(ENV_FILE, override=True)
session = boto3.Session()
agentcore = session.client("bedrock-agentcore-control")
memory_records = session.client("bedrock-agentcore")
lambda_client = session.client("lambda")

app_url = lambda_client.get_function_url_config(FunctionName=APP_FUNCTION)["FunctionUrl"].rstrip("/")
LAB_MEMORY_ID = os.environ["UNDERCOVER_MEMORY_ID"]

print(f"App on Lambda: {app_url}")
print(f"Gateway: {os.environ['UNDERCOVER_GATEWAY_URL']}")
print(f"The lab's Memory, put back at the end: {LAB_MEMORY_ID}")

### One adjustment for Windows

The agent runs as a separate program, so the `truststore` line above doesn't reach it. Behind a network that
inspects HTTPS traffic, its calls fail with "certificate verify failed". On Windows this cell saves the certificates
Windows trusts to a file, and the agent's runs below are pointed at it. On Mac and Linux it does nothing.

In [ ]:
import sys
import tempfile

CERT_ENV = {}

if sys.platform == "win32":
    import ssl

    certs = Path(tempfile.gettempdir()) / "undercover_windows_certs.pem"
    pems = {ssl.DER_cert_to_PEM_cert(cert) for store in ("ROOT", "CA") for cert, encoding, _ in ssl.enum_certificates(store) if encoding == "x509_asn"}
    certs.write_text("".join(pems))
    CERT_ENV = {"SSL_CERT_FILE": str(certs), "REQUESTS_CA_BUNDLE": str(certs), "AWS_CA_BUNDLE": str(certs)}
    print("Applied the Windows adjustment")
else:
    print("Not Windows, so nothing to do here")

## A Memory with three strategies

The lab's two strategies stay as they were. The third is new:

- **Semantic** (`/customers/{actorId}/facts/`): facts about the customer.
- **Summary** (`/customers/{actorId}/claims/{sessionId}/`): a summary of each claim.
- **Episodic** (`/customers/{actorId}/episodes/{sessionId}/`): each claim as an episode, the situation, what the
  agent did and how it turned out. Its **reflections** go one level up, in `/customers/{actorId}/episodes/`, because
  a reflection draws lessons across a customer's episodes, not from one claim.

All three sit under `/customers/{actorId}`, so the agent's `search_memory` tool, which searches that path, sees the
episodes and reflections too, and still nothing of another customer.

Creating a Memory takes two to three minutes.

In [ ]:
def find_episodic_memories():
    """The memories named undercover_episodic, leaving out any being deleted."""
    pages = agentcore.get_paginator("list_memories").paginate()
    return [m for page in pages for m in page["memories"] if m["id"].startswith(EPISODIC_MEMORY_NAME) and m["status"] != "DELETING"]


existing = find_episodic_memories()

if existing:
    memory_id = existing[0]["id"]
else:
    memory_id = agentcore.create_memory(
        name=EPISODIC_MEMORY_NAME,
        description="The UnderCover Agent's memory of each customer, with episodes",
        eventExpiryDuration=7,
        memoryStrategies=[
            {"semanticMemoryStrategy": {"name": "customer_facts", "namespaceTemplates": ["/customers/{actorId}/facts/"]}},
            {"summaryMemoryStrategy": {"name": "claim_summaries", "namespaceTemplates": ["/customers/{actorId}/claims/{sessionId}/"]}},
            {
                "episodicMemoryStrategy": {
                    "name": "claim_episodes",
                    "namespaceTemplates": ["/customers/{actorId}/episodes/{sessionId}/"],
                    "reflectionConfiguration": {"namespaceTemplates": ["/customers/{actorId}/episodes/"]},
                }
            },
        ],
        tags=PROJECT_TAG,
    )["memory"]["id"]

while True:
    memory = agentcore.get_memory(memoryId=memory_id)["memory"]
    if memory["status"] != "CREATING":
        break
    time.sleep(10)

strategy_type = {s["strategyId"]: s["type"] for s in memory["strategies"]}
print(f"Memory {memory_id}: {memory['status']}")
for strategy in memory["strategies"]:
    print(f"  {strategy['type']:<15}{', '.join(strategy['namespaces'])}")

### Pointing the agent at it

The agent reads `.env` with `override=True` when it starts, so an environment variable set here wouldn't reach it.
This cell writes the new Memory into `.env`; the teardown puts the lab's Memory back.

In [ ]:
set_key(ENV_FILE, "UNDERCOVER_MEMORY_ID", memory_id, quote_mode="never")
print(f"UNDERCOVER_MEMORY_ID is now {memory_id}")

## Oliver Grant's two claims again

### A fresh queue

As in Part 5, reseed so both claims are waiting: the local database, and RDS too if you took the AWS route in place
of Supabase.

In [ ]:
def run(command, **environment):
    """Run a command from the repo root, print the end of its output, and stop on failure."""
    # Leave out the notebook's VIRTUAL_ENV, so uv uses the repo's .venv without a warning.
    base = {key: value for key, value in os.environ.items() if key != "VIRTUAL_ENV"}
    result = subprocess.run(command, cwd=REPO, env=base | CERT_ENV | environment, capture_output=True, text=True, check=False)
    print((result.stdout + result.stderr).strip()[-3000:])
    result.check_returncode()


uv = shutil.which("uv")
run([uv, "run", "--env-file", ".env", "-m", "backend.seed"])
if os.getenv("AWS_DATABASE_URL"):
    run([uv, "run", "-m", "backend.seed"], DATABASE_URL=os.environ["AWS_DATABASE_URL"])

### The helpers

`work` runs the agent on one claim, as Part 5 does. `records` gathers every memory record AgentCore has made for a
customer, with the strategy that made it and the namespace it's in. It lists them under the customer's path, and if
the listing comes back empty it asks each strategy through a search instead, the same call the agent's
`search_memory` tool makes. `wait_until` waits for AgentCore to extract them in the background.

In [ ]:
def work(claim_id):
    """Work one claim with the agent on your laptop, against the app on Lambda."""
    run([uv, "run", "-m", "agent", claim_id], UNDERCOVER_API=app_url)


def records(customer_id=CUSTOMER):
    """Every memory record for one customer, as (strategy, namespace, text), oldest first."""
    path = f"/customers/{customer_id}"
    found = memory_records.list_memory_records(memoryId=memory_id, namespacePath=path, maxResults=100)["memoryRecordSummaries"]
    if not found:
        for strategy_id in strategy_type:
            found += memory_records.retrieve_memory_records(
                memoryId=memory_id,
                namespacePath=path,
                searchCriteria={"searchQuery": "the customer's claims", "memoryStrategyId": strategy_id, "topK": 100},
            )["memoryRecordSummaries"]
    unique = {r["memoryRecordId"]: r for r in found}.values()
    ordered = sorted(unique, key=lambda r: r["createdAt"])
    return [(strategy_type.get(r["memoryStrategyId"], r["memoryStrategyId"]), r["namespaces"][0], r["content"]["text"]) for r in ordered]


def count(kind, mentioning=""):
    """How many of a strategy's records there are, optionally only those mentioning some text."""
    return sum(1 for strategy, namespace, text in records() if strategy == kind and mentioning in namespace + text)


def wait_until(label, ready, minutes=10):
    """Check every 20 seconds until ready() is true, or give up after some minutes."""
    started = time.time()
    while time.time() - started < minutes * 60:
        if ready():
            print(f"{label}: ready after {time.time() - started:.0f}s")
            return
        time.sleep(20)
    print(f"{label}: not ready after {minutes} minutes")


def show():
    """Print every record for the customer, with its strategy and namespace."""
    for strategy, namespace, text in records():
        print(f"[{strategy}] {namespace}\n{text}\n")

### The first claim

The giraffe in Nairobi. The agent saves two events when it has decided, the claim and the decision, and AgentCore
extracts the memories in the background.

In [ ]:
work("CLM-1005")

In [ ]:
wait_until("Facts", lambda: count("SEMANTIC") > 0)
wait_until("Summary of CLM-1005", lambda: count("SUMMARIZATION", "CLM-1005") > 0)
wait_until("Episode of CLM-1005", lambda: count("EPISODIC", "CLM-1005") > 0)
show()

### The second claim

The coconut in Sri Lanka, which opens "Yes, it is me again". The agent decides for itself whether to search its
memory, and that search now reaches the first claim's episode as well.

In [ ]:
work("CLM-1006")

In [ ]:
episodic_before = count("EPISODIC")
wait_until("Summary of CLM-1006", lambda: count("SUMMARIZATION", "CLM-1006") > 0)
wait_until("Episode of CLM-1006", lambda: count("EPISODIC", "CLM-1006") > 0)
# A reflection draws on more than one episode, so give it a few more minutes to update.
wait_until("New episodic records", lambda: count("EPISODIC") > episodic_before + 1, minutes=5)

## The comparison

### Everything kept about Oliver, by strategy

In [ ]:
everything = records()

for kind in ["SEMANTIC", "SUMMARIZATION", "EPISODIC"]:
    kept = [(namespace, text) for strategy, namespace, text in everything if strategy == kind]
    words = sum(len(text.split()) for _, text in kept)
    print(f"===== {kind}: {len(kept)} records, {words} words\n")
    for namespace, text in kept:
        print(f"{namespace}\n{text}\n")

### What the agent's search finds

The same search as the agent's `search_memory` tool: everything under the customer's path, top 5.

In [ ]:
def search_memory(customer_id, query):
    """Search one customer's memories, as the agent's search_memory tool does."""
    response = memory_records.retrieve_memory_records(
        memoryId=memory_id,
        namespacePath=f"/customers/{customer_id}",
        searchCriteria={"searchQuery": query, "topK": 5},
    )
    return [(strategy_type.get(r["memoryStrategyId"], "?"), r["content"]["text"]) for r in response["memoryRecordSummaries"]]


for query in ["earlier claims", "how was this customer's last claim decided"]:
    print(f"===== {query}")
    for strategy, text in search_memory(CUSTOMER, query):
        print(f"[{strategy}] {text[:300]}\n")
print(f"CUS-3001 finds {len(search_memory('CUS-3001', 'earlier claims'))} memories")

## What I found

From one run on 3 November 2026: CLM-1005 approved for $243.00 under section 6, CLM-1006 approved for $425.00 under
section 7. The numbers below are from that run's outputs.

**What each strategy kept about Oliver**

| Strategy | Records | Words | What it is |
|---|---|---|---|
| Semantic | 6 | 137 | Three one-line facts per claim: what happened, what was claimed, what was paid |
| Summary | 4 | 474 | Two summaries per claim, one as a `<topic>` list and one as markdown tables, saying much the same |
| Episodic | 2 | 555 | One episode, for CLM-1005, and one reflection across the customer's episodes |

**What Episodic keeps that the others don't**

1. **How the agent worked, not just what it decided.** The episode records the situation, the intent, the action,
   the agent's reasoning and a self-assessment: it itemised the three costs, mapped them to section 6, checked the
   $243 total against the $250 cap and judged the lodge letter and receipts sufficient. Facts and summaries keep the
   outcome; the episode keeps the method.
2. **A lesson.** The reflection, titled "Structured Insurance Claim Evaluation Against Policy Sections", turns the
   episode into five hints with a confidence of 0.8. The most interesting one: an implausible story, like a giraffe
   eating a passport, shouldn't derail the decision, which should rest on the evidence and the policy.
3. **It wins the "how" question.** For "how was this customer's last claim decided", the agent's search ranked the
   episode and the reflection first and second. For "earlier claims" the summaries and facts came first.

**What surprised me**

1. **Episodes are slow.** Facts arrived in about 68 seconds and each summary in about a minute. The CLM-1005 episode
   took more than 10 minutes, and the CLM-1006 episode still hadn't appeared after about 12 minutes, when the run ended.
   The agent works a returning customer's next claim within minutes, so it can't count on the last episode being there.
2. **Why the second episode was missing is my guess, not checked.** The CLM-1005 episode marks the user's side as not
   finished ("no clear signal that the user's inquiry has concluded"). Our sessions are only two events long, a claim
   and a decision, so AgentCore may wait for more before it closes an episode.
3. **The reflection isn't about Oliver.** Its hints are about judging any passport claim, yet it's filed under
   `/customers/CUS-3005/episodes/`, so only Oliver's future claims can find it. Reflections would be more useful one
   level up, shared across customers, but then the agent's search, which stays inside one customer's path, couldn't
   reach them, and the namespace no longer keeps customers apart.
4. **Summary duplicates itself.** Each claim got two summaries of the same content in two formats, which doubles what
   the agent reads back on every search.

**What I'd do in production**

- Keep Semantic for facts about the customer and Summary for each claim, as the lab does.
- Use Episodic for the agent's own lessons, under a namespace shared across customers, searched by a separate tool,
  and keep customer details out of what it stores there.
- Don't rely on episodes for a customer's very next claim; they arrive too late.

## Teardown

Puts the lab's Memory back in `.env` and deletes the `undercover_episodic` Memory. The lab's own Part 6 teardown
deletes the rest. Safe to run again.

In [ ]:
set_key(ENV_FILE, "UNDERCOVER_MEMORY_ID", LAB_MEMORY_ID, quote_mode="never")
print(f"UNDERCOVER_MEMORY_ID is back to {LAB_MEMORY_ID}")

for memory_summary in find_episodic_memories():
    agentcore.delete_memory(memoryId=memory_summary["id"])
    print(f"Deleted memory {memory_summary['id']}")